In [19]:
import sys, json, joblib 
from pathlib import Path 
import numpy as np
import pandas as pd 

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data.cicids_preprocessing import load_processed_data

(X_train, X_test, y_train, y_test,
 labels_train, labels_test, preprocessor) = load_processed_data(
    ROOT / "data" / "processed" / "cicids2017")

x_normal = X_train[(y_train == 0).to_numpy()]
print(X_train.shape, X_test.shape, x_normal.shape)



(2262296, 70) (565580, 70) (1817051, 70)


In [11]:
from sklearn.covariance import LedoitWolf
mu = x_normal.mean(axis=0, dtype=np.float64)
cov_inv = LedoitWolf().fit(x_normal).precision_

def mahalanobis_sq(X, mu, cov_inv, chunk=200_000):
    out = np.empty(len(X))
    for i in range(0, len(X), chunk):
        d = X[i:i + chunk] - mu
        out[i:i + chunk] = ((d @ cov_inv) * d).sum(axis=1)
    return out

/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/scipy/linalg/_basic.py:1542: RuntimeWarning: divide by zero encountered in matmul
  B = (u * psigma_diag) @ u.conj().T
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/scipy/linalg/_basic.py:1542: RuntimeWarning: overflow encountered in matmul
  B = (u * psigma_diag) @ u.conj().T
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/scipy/linalg/_basic.py:1542: RuntimeWarning: invalid value encountered in matmul
  B = (u * psigma_diag) @ u.conj().T


In [12]:
from sklearn.metrics import precision_score, recall_score

d2_normal = mahalanobis_sq(x_normal, mu, cov_inv)
threshold_mah = np.percentile(d2_normal, 95)
print("threshold:", threshold_mah)

d2_test = mahalanobis_sq(X_test, mu, cov_inv)
pred = (d2_test > threshold_mah).astype(int)
print("flagged:", pred.sum(), "of", len(pred))
print("precision:", precision_score(y_test, pred))
print("recall:", recall_score(y_test, pred))

/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_35477/2039012349.py:9: RuntimeWarning: divide by zero encountered in matmul
  out[i:i + chunk] = ((d @ cov_inv) * d).sum(axis=1)
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_35477/2039012349.py:9: RuntimeWarning: overflow encountered in matmul
  out[i:i + chunk] = ((d @ cov_inv) * d).sum(axis=1)
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_35477/2039012349.py:9: RuntimeWarning: invalid value encountered in matmul
  out[i:i + chunk] = ((d @ cov_inv) * d).sum(axis=1)


threshold: 91.19617761071852
flagged: 74495 of 565580
precision: 0.6927176320558427
recall: 0.4636019800379118


In [13]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(random_state=42, n_jobs=-1).fit(x_normal)

def if_score(X, chunk=200_000):
    return np.concatenate([-iso.score_samples(X[i:i + chunk]) for i in range(0, len(X), chunk)])

s_normal = if_score(x_normal)
threshold_if = np.percentile(s_normal, 95)

s_test = if_score(X_test)
pred = (s_test > threshold_if).astype(int)

print("threshold:", threshold_if)
print("flagged:", pred.sum(), "of", len(pred))
print("precision:", precision_score(y_test, pred))
print("recall:", recall_score(y_test, pred))

threshold: 0.5345972095659743
flagged: 57484 of 565580
precision: 0.6070906687078144
recall: 0.3135179811519077


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class Autoencoder(nn.Module):
    def __init__(self, n_features=70, bottleneck=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, n_features),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(42)
model_ae = Autoencoder(n_features=x_normal.shape[1], bottleneck=16)

rng = np.random.default_rng(42)
idx = rng.permutation(len(x_normal))
n_val = int(0.10 * len(x_normal))
val_rows, train_rows = idx[:n_val], idx[n_val:]

x_tr = torch.tensor(x_normal[train_rows], dtype=torch.float32)
x_val = torch.tensor(x_normal[val_rows], dtype=torch.float32)
train_loader = DataLoader(TensorDataset(x_tr), batch_size=256, shuffle=True)

optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
best_val, patience, wait, best_state = float("inf"), 5, 0, None

for epoch in range(1, 101):
    model_ae.train()
    for (batch,) in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model_ae(batch), batch)
        loss.backward()
        optimizer.step()

    model_ae.eval()
    with torch.no_grad():
        val_loss = loss_fn(model_ae(x_val), x_val).item()
    print(f"epoch {epoch:3d}  val loss {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val, wait = val_loss, 0
        best_state = {k: v.clone() for k, v in model_ae.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            print("early stop")
            break

model_ae.load_state_dict(best_state)

epoch   1  val loss 0.1596
epoch   2  val loss 0.1105
epoch   3  val loss 0.1195
epoch   4  val loss 0.1311
epoch   5  val loss 0.0896
epoch   6  val loss 0.2611
epoch   7  val loss 0.0969
epoch   8  val loss 0.1177
epoch   9  val loss 0.0621
epoch  10  val loss 0.0477
epoch  11  val loss 0.0570
epoch  12  val loss 0.0552
epoch  13  val loss 0.0773
epoch  14  val loss 0.0227
epoch  15  val loss 0.0450
epoch  16  val loss 0.0509
epoch  17  val loss 0.0421
epoch  18  val loss 0.0415
epoch  19  val loss 0.0394
early stop


<All keys matched successfully>

In [15]:
def recon_error(model, X, chunk=200_000):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), chunk):
            t = torch.tensor(X[i:i + chunk], dtype=torch.float32)
            out.append(((model(t) - t) ** 2).mean(dim=1).numpy())
    return np.concatenate(out)

err_val = recon_error(model_ae, x_normal[val_rows])
err_test = recon_error(model_ae, X_test)

threshold_ae = np.percentile(err_val, 95)
pred = (err_test > threshold_ae).astype(int)

print("threshold:", threshold_ae)
print("flagged:", pred.sum(), "of", len(pred))
print("precision:", precision_score(y_test, pred))
print("recall:", recall_score(y_test, pred))

threshold: 0.027485244
flagged: 74147 of 565580
precision: 0.6901425546549422
recall: 0.4597209619893811


In [20]:
maha_scores, maha_thr = d2_test, 91.19617761071852      # edit names if yours differ
iso_scores, iso_thr = s_test, threshold_if
ae_scores, ae_thr = err_test, threshold_ae

labels_test = joblib.load("../data/processed/cicids2017/attack_labels_test.joblib")
res = pd.DataFrame({
    "label": np.asarray(labels_test).astype(str),
    "Mahalanobis": maha_scores > maha_thr,
    "Isolation Forest": iso_scores > iso_thr,
    "Autoencoder": ae_scores > ae_thr,
})
table = res.groupby("label").agg(rows=("Mahalanobis", "size"), Mahalanobis=("Mahalanobis", "mean"),
                                 IsolationForest=("Isolation Forest", "mean"), Autoencoder=("Autoencoder", "mean"))
print(table.sort_values("rows", ascending=False).round(3).to_string())

                              rows  Mahalanobis  IsolationForest  Autoencoder
label                                                                        
BENIGN                      454269        0.050            0.050        0.051
DoS Hulk                     46025        0.701            0.618        0.676
PortScan                     31761        0.007            0.001        0.003
DDoS                         25605        0.625            0.180        0.638
DoS GoldenEye                 2059        0.691            0.189        0.903
FTP-Patator                   1587        0.011            0.000        0.001
SSH-Patator                   1179        0.000            0.000        0.000
DoS slowloris                 1159        0.548            0.494        0.564
DoS Slowhttptest              1100        0.905            0.752        0.965
Bot                            391        0.026            0.015        0.015
Web Attack - Brute Force       302        0.003            0.053

In [21]:
import os, json, shutil, sys
sys.path.append("..")
from src.detection.detectors import load_all

A = "../artifacts/cicids2017"
os.makedirs(A, exist_ok=True)

joblib.dump(iso, f"{A}/iso_forest.joblib")
np.savez(f"{A}/mahalanobis.npz", mu=mu, cov_inv=cov_inv)
torch.save(model_ae.state_dict(), f"{A}/autoencoder.pt")
shutil.copy("../data/processed/cicids2017/preprocessor.joblib", f"{A}/preprocessor.joblib")

thr = {
    "autoencoder": float(threshold_ae),
    "mahalanobis": float(maha_thr),
    "isolation_forest": float(threshold_if),
}
json.dump(thr, open(f"{A}/thresholds.json", "w"), indent=2)
print(os.listdir(A))
print(thr)

# round trip: the API's own loader must flag exactly the rows this notebook flagged
models = load_all(A)
n = 20_000
for key, scores, t in (("autoencoder", err_test, threshold_ae),
                       ("mahalanobis", maha_scores, maha_thr),
                       ("isolation_forest", s_test, threshold_if)):
    out = models[key].predict(X_test[:n])
    print(key, "same flags:", bool((out["flagged"] == (scores[:n] > t)).all()),
          "| max score diff:", float(np.abs(out["score"] - scores[:n]).max()))

['autoencoder.pt', 'preprocessor.joblib', 'thresholds.json', 'mahalanobis.npz', 'iso_forest.joblib']
{'autoencoder': 0.02748524397611618, 'mahalanobis': 91.19617761071852, 'isolation_forest': 0.5345972095659743}
autoencoder same flags: True | max score diff: 0.0
mahalanobis same flags: True | max score diff: 0.0
isolation_forest same flags: True | max score diff: 0.0


/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: divide by zero encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: overflow encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
/Users/sayoowo5/CS4360-Senior-Experience-Project/src/detection/detectors.py:62: RuntimeWarning: invalid value encountered in matmul
  return ((d @ self.cov_inv) * d).sum(axis=1)
